# ASMSA v2 &mdash; Setup

&#8592; [Quickstart](00-quickstart.ipynb) &nbsp;|&nbsp; [Prepare](02-prepare.ipynb) &#8594;

| | |
|---|---|
| **What it does** | Describe your system once, in a validated config file every later notebook reads. |
| **Typical runtime** | under a minute |
| **Needs** | a conformation (PDB, no hydrogens) and a trajectory &mdash; or nothing, to use the demo system |
| **Produces** | <code>asmsa.yaml</code> and the stage directory layout |

---

Everything downstream reads the `Project` written here, so the inputs are validated
**now** rather than failing five notebooks later.

> *Note on v1:* the original pipeline passed state between notebooks by writing a Python
> file and `exec`-ing it (`exec(open('inputs.py').read())`). A typo there surfaced as a
> `NameError` in a much later notebook, and nothing checked that the files it named
> existed or were mutually consistent. `Project` replaces that with a validated YAML
> config plus a preflight check.

In [ ]:
from pathlib import Path

import asmsa_v2 as asmsa
from asmsa_v2.viz import use_asmsa_style

use_asmsa_style()

# Working directory for this system. Everything the pipeline produces lands here.
WORKDIR = Path('run').resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
print('working directory:', WORKDIR)

## 1. Choose your input

Run **either** 1a (demo) **or** 1b (your own data), not both.

### 1a. Demo system

A synthetic 12-residue peptide + 5-atom ligand with two binding poses and one slow
breathing mode. It is a *geometric* mock, not a force-field simulation &mdash; use it to
learn and test the pipeline, never for physical conclusions. It makes notebooks 01&ndash;05
and 07 runnable without any data or GROMACS installed.

In [ ]:
USE_DEMO = False   # set False if you are supplying your own files in 1b

if USE_DEMO:
    from asmsa_v2.demo import make_demo_complex
    demo = make_demo_complex(WORKDIR / 'inputs', n_residues=12,
                             n_frames=3000, seed=0)
    CONF = str(demo.conf.relative_to(WORKDIR))
    TRAJ = str(demo.traj.relative_to(WORKDIR))
    COMPONENTS = {'protein': 'not resname LIG', 'ligand': 'resname LIG'}
    NAME = 'demo'
    print(demo.summary())

### 1b. Your own system

| Field | Requirement |
|---|---|
| `CONF` | A PDB of one conformation. **No hydrogens** &mdash; `gmx pdb2gmx` adds them in [02-prepare](02-prepare.ipynb), and letting it do so is what keeps the PDB, topology and `.gro` mutually consistent. |
| `TRAJ` | A trajectory whose atom count **and order** match `CONF` exactly. |
| `COMPONENTS` | [mdtraj selection strings](https://mdtraj.org/latest/atom_selection.html) naming the parts of the system. Leave as `{}` for a single molecule, which reproduces v1's behaviour. |

For a protein-ligand complex both components must be given: the inter-component
distances generated from them are the features that encode the binding pose.

In [ ]:
if not USE_DEMO:
    NAME = 'trpcage'                      # short identifier, used as a file stem
    CONF = '/home/tedeschg/prj/unfolding-data/trpcage_npt400_nH.pdb'           # relative to WORKDIR, or absolute
    TRAJ = '/home/tedeschg/prj/unfolding-data/trpcage_ds_nH.xtc'

    # Protein-ligand complex:
    # COMPONENTS = {'protein': 'protein', 'ligand': 'resname LIG'}
    # Protein-protein complex:
    # COMPONENTS = {'chainA': 'chainid 0', 'chainB': 'chainid 1'}
    # Single molecule (v1 behaviour):
    COMPONENTS = {}

## 2. Build the project

A few choices worth understanding before you accept the defaults:

| Setting | Default | Why |
|---|---|---|
| `latent_dim` | `2` | The number of collective variables. Metadynamics must grid the CV space, so cost grows steeply with dimension; 2 is also the only choice you can simply *look at*. |
| `prior` | `normal` | The shape the latent is pushed towards. `mix` (Gaussians on a circle) encourages discrete, well-separated states; `uniform` spreads frames evenly. |
| `split_strategy` | `chronological` | Consecutive MD frames are near-identical, so a *random* split leaks training structures into validation and the validation loss then measures memorization. Contiguous blocks confine the leak to two boundaries. |
| `time_lag` | `0` | `0` reconstructs the input frame. Set it to a positive number of frames to reconstruct the *future* frame instead, which makes the latent track the slow motions rather than the loudest ones &mdash; usually what you want for enhanced-sampling CVs. |
| `scaler_method` | `standard` | Mean/standard-deviation standardization. Use `robust` (median/IQR) if a handful of frames contain unfolded or dissociated geometries that would otherwise inflate every feature's scale. |

In [ ]:
project = asmsa.Project(
    name=NAME,
    root=WORKDIR,
    conf=CONF,
    traj=TRAJ,
    components=COMPONENTS,

    # --- model ---
    latent_dim=2,
    prior='uniform',          # 'normal' | 'uniform' | 'mix' | path to a grayscale image
    model_name='norm',       # names the trained model + its exported artifacts
    time_lag=0,              # >0 switches on the time-lagged objective

    # --- featurization ---
    inter_strategy='ca_heavy',   # 'ca_heavy' | 'contact' | 'all'
    use_dihedrals=True,
    scaler_method='standard',

    # --- splitting ---
    split_strategy='random',
    split_fractions=(0.7, 0.15, 0.15),
    rmsd_decorrelate_nm=None,    # e.g. 0.05 to drop train frames too close to held-out ones

    # --- MD (used from 06-md onwards) ---
    temperature=300.0,
    pressure=1.0,
    dt=0.002,
    box_nm=1.2,
    production_ns=100.0,
)
project.ensure_dirs()
config_path = project.save()
print('wrote', config_path)
project

## 3. Preflight check

Validates the inputs and reports the environment. Read it before going on.

- **red (&#10007;)** must be fixed &mdash; later notebooks will fail.
- **amber (&#9888;)** is informational: an optional package is missing, or the
  conformation contains hydrogens (fine if you know why; otherwise see 1b).
- GROMACS and PLUMED are only needed from [06-md](06-md.ipynb) onwards, so they are not
  required to be present yet.

In [ ]:
report = project.check()
report

In [ ]:
# Uncomment to make a failed check stop the notebook rather than be scrolled past.
# report.raise_if_failed()

## 4. Look at the structure

The single most effective check available. Because PDB and trajectory formats number
atoms by different conventions, a mismatch between `conf` and `traj` is common &mdash;
and it shows up here as implausible bond lengths or a visibly mangled molecule, while
every numerical check above still passes.

**What to look for:** a connected, sensible structure, and the two components in
distinct colours sitting next to each other rather than interpenetrating.

In [ ]:
import mdtraj as md

traj = md.load(str(project.traj_path), top=str(project.conf_path))
align = traj.top.select('name CA')
if len(align) < 3:
    align = traj.top.select('element != H')
traj.superpose(traj, frame=0, atom_indices=align)

print(f'{traj.n_frames} frames, {traj.n_atoms} atoms, {traj.n_residues} residues')
for cname, sel in (project.components or {'system': 'all'}).items():
    print(f'  {cname:<10} {len(traj.top.select(sel)):>5} atoms   ({sel})')

In [ ]:
try:
    from asmsa_v2.viz.structure import view_frame
    comp_atoms = {k: traj.top.select(v) for k, v in (project.components or {}).items()}
    view = view_frame(
        traj, 0, style='sticks',
        component_atoms=comp_atoms or None,
        component_colors={k: c for k, c in zip(comp_atoms, ['cyan', 'magenta', 'yellow'])}
        or None,
    )
    view.show()
except ImportError as exc:
    print('3-D viewer unavailable:', exc)
    print('install it with:  pip install py3Dmol')

### Bond-length sanity check

A cheap numerical stand-in for the visual check, and a decisive one: covalent bonds in
a correctly ordered structure are 0.10&ndash;0.16 nm. A mean far outside that range
almost always means the trajectory and the PDB disagree about atom order.

In [ ]:
import numpy as np

pairs = np.array([[b.atom1.index, b.atom2.index] for b in traj.top.bonds])
if len(pairs):
    d = md.compute_distances(traj[:100], pairs)
    print(f'{len(pairs)} bonds: mean {d.mean():.3f} nm, '
          f'range {d.min():.3f}-{d.max():.3f} nm')
    verdict = 'looks right' if 0.09 < d.mean() < 0.20 else 'SUSPICIOUS'
    print(f'  -> {verdict}  (expect roughly 0.10-0.16 nm)')
else:
    print('no bonds in the topology; skipping (normal for a bare .gro)')

## Re-opening this project later

Every later notebook starts with exactly this, so the configuration lives in one place:

```python
import asmsa_v2 as asmsa
project = asmsa.Project.load('run')
```

---

&#8592; [Quickstart](00-quickstart.ipynb) &nbsp;|&nbsp; **Next: [Prepare](02-prepare.ipynb)** &#8594;